### Booking Cancellation Model (CatBoost + Probability Calibration)

Notebook này train model dự báo khả nănng hủy booking từ `hotel_bookings.csv`, calibrate xác suất để output phần trăm ổn định hơn, và lưu artifact để deploy.

In [3]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import average_precision_score, brier_score_loss, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

SEED = 42
DATA_PATH = Path('../data/hotel_bookings.csv')
ARTIFACT_DIR = Path('../models')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LEAKAGE_COLUMNS = ['reservation_status', 'reservation_status_date']
TARGET_COLUMN = 'is_canceled'

In [4]:
df = pd.read_csv(DATA_PATH)
print('Raw shape:', df.shape)
print('Target rate (is_canceled=1):', round(df[TARGET_COLUMN].mean(), 4))
df.head()

Raw shape: (119390, 32)
Target rate (is_canceled=1): 0.3704


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [5]:
missing_cols = [col for col in LEAKAGE_COLUMNS + [TARGET_COLUMN] if col not in df.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

feature_columns = [c for c in df.columns if c not in LEAKAGE_COLUMNS + [TARGET_COLUMN]]
X = df[feature_columns].copy()
y = df[TARGET_COLUMN].astype(int).copy()

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=SEED, stratify=y_temp
)

print('Train:', X_train.shape, 'Valid:', X_valid.shape, 'Test:', X_test.shape)

Train: (71634, 29) Valid: (23878, 29) Test: (23878, 29)


In [6]:
categorical_cols = X_train.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
numerical_cols = [c for c in feature_columns if c not in categorical_cols]

numeric_medians = X_train[numerical_cols].apply(pd.to_numeric, errors='coerce').median().to_dict()

def preprocess(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()

    for col in feature_columns:
        if col not in out.columns:
            out[col] = pd.NA

    out = out[feature_columns]

    for col in categorical_cols:
        out[col] = out[col].astype('string').fillna('missing')

    for col in numerical_cols:
        out[col] = pd.to_numeric(out[col], errors='coerce').fillna(numeric_medians[col])

    return out

X_train_p = preprocess(X_train)
X_valid_p = preprocess(X_valid)
X_test_p = preprocess(X_test)

print(f'Categorical: {len(categorical_cols)} cols')
print(f'Numerical: {len(numerical_cols)} cols')

Categorical: 10 cols
Numerical: 19 cols


In [7]:
base_model = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='AUC',
    iterations=2500,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=6,
    random_seed=SEED,
    auto_class_weights='Balanced',
    verbose=200
)

base_model.fit(
    X_train_p,
    y_train,
    cat_features=categorical_cols,
    eval_set=(X_valid_p, y_valid),
    use_best_model=True,
    early_stopping_rounds=150,
)

raw_test_proba = base_model.predict_proba(X_test_p)[:, 1]

0:	test: 0.8578639	best: 0.8578639 (0)	total: 98.7ms	remaining: 4m 6s
200:	test: 0.9434303	best: 0.9434303 (200)	total: 1m 3s	remaining: 12m 48s
400:	test: 0.9482656	best: 0.9482656 (400)	total: 1m 52s	remaining: 10m 24s
600:	test: 0.9507025	best: 0.9507025 (600)	total: 2m 38s	remaining: 8m 46s
800:	test: 0.9522365	best: 0.9522370 (799)	total: 3m 38s	remaining: 8m 7s
1000:	test: 0.9532663	best: 0.9532663 (1000)	total: 4m 20s	remaining: 6m 49s
1200:	test: 0.9540308	best: 0.9540333 (1199)	total: 5m 6s	remaining: 5m 47s
1400:	test: 0.9545586	best: 0.9545608 (1399)	total: 6m 6s	remaining: 5m 1s
1600:	test: 0.9549683	best: 0.9549683 (1600)	total: 6m 39s	remaining: 3m 54s
1800:	test: 0.9554789	best: 0.9554795 (1799)	total: 7m 16s	remaining: 2m 57s
2000:	test: 0.9558066	best: 0.9558066 (2000)	total: 7m 50s	remaining: 2m 2s
2200:	test: 0.9560813	best: 0.9560834 (2195)	total: 8m 36s	remaining: 1m 13s
2400:	test: 0.9562447	best: 0.9562525 (2389)	total: 9m 1s	remaining: 23.2s
2499:	test: 0.956334

In [8]:
# Probability calibration on validation set
try:
    from sklearn.frozen import FrozenEstimator

    calibrator = CalibratedClassifierCV(
        estimator=FrozenEstimator(base_model),
        method='sigmoid',
    )
except Exception:
    calibrator = CalibratedClassifierCV(
        estimator=base_model,
        method='sigmoid',
        cv='prefit',
    )

calibrator.fit(X_valid_p, y_valid)
calibrated_test_proba = calibrator.predict_proba(X_test_p)[:, 1]

In [9]:
metrics = {
    'raw_roc_auc': roc_auc_score(y_test, raw_test_proba),
    'calibrated_roc_auc': roc_auc_score(y_test, calibrated_test_proba),
    'raw_pr_auc': average_precision_score(y_test, raw_test_proba),
    'calibrated_pr_auc': average_precision_score(y_test, calibrated_test_proba),
    'raw_brier': brier_score_loss(y_test, raw_test_proba),
    'calibrated_brier': brier_score_loss(y_test, calibrated_test_proba),
}

print('=== Test Metrics ===')
for k, v in metrics.items():
    print(f'{k}: {v:.6f}')

pred_label = (calibrated_test_proba >= 0.5).astype(int)
print('\n=== Classification Report @ 0.50 ===')
print(classification_report(y_test, pred_label, digits=4))

=== Test Metrics ===
raw_roc_auc: 0.956895
calibrated_roc_auc: 0.956895
raw_pr_auc: 0.936203
calibrated_pr_auc: 0.936203
raw_brier: 0.082312
calibrated_brier: 0.081447

=== Classification Report @ 0.50 ===
              precision    recall  f1-score   support

           0     0.9090    0.9109    0.9099     15033
           1     0.8480    0.8450    0.8465      8845

    accuracy                         0.8865     23878
   macro avg     0.8785    0.8779    0.8782     23878
weighted avg     0.8864    0.8865    0.8864     23878



In [10]:
base_model_path = ARTIFACT_DIR / 'booking_cancel_catboost.cbm'
calibrator_path = ARTIFACT_DIR / 'booking_cancel_calibrator.pkl'
meta_path = ARTIFACT_DIR / 'booking_cancel_meta.json'

base_model.save_model(base_model_path)
joblib.dump(calibrator, calibrator_path)

meta = {
    'target': TARGET_COLUMN,
    'feature_columns': feature_columns,
    'categorical_columns': categorical_cols,
    'numerical_columns': numerical_cols,
    'numeric_medians': {k: float(v) for k, v in numeric_medians.items()},
    'metrics': {k: float(v) for k, v in metrics.items()},
    'decision_threshold': 0.5,
}

with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print('Saved:', base_model_path)
print('Saved:', calibrator_path)
print('Saved:', meta_path)

Saved: ../models/booking_cancel_catboost.cbm
Saved: ../models/booking_cancel_calibrator.pkl
Saved: ../models/booking_cancel_meta.json


In [11]:
def predict_cancel_probability(payload: dict) -> float:
    row = pd.DataFrame([payload])
    row_p = preprocess(row)
    return float(calibrator.predict_proba(row_p)[:, 1][0])

sample = X_test.iloc[0].to_dict()
print('Sample cancellation probability:', round(predict_cancel_probability(sample), 4))

Sample cancellation probability: 0.0698
